# Extracción de mapas de pooling (16×16) — Colab GPU

Genera `results/pooling_maps.csv`: los pesos por token visual de las **8 técnicas de pooling** para las 129 imágenes (capa 34, prompt P1). El dashboard los renderiza como heatmaps superpuestos al fundus.

**Antes de empezar:**
1. Runtime → Change runtime type → **GPU (T4)**.
2. En Colab, panel izquierdo → 🔑 Secrets → añade `HF_TOKEN` (con licencia HAI-DEF de MedGemma aceptada).
3. Comprime la carpeta del proyecto (al menos `src/`, `config.yaml`, `requirements.txt`) como `proyecto.zip` — la celda 2 te pedirá subirlo.

Tiempo estimado: ~15–25 min (129 imágenes, con atenciones en modo eager).

In [ ]:
# 1) Dependencias
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q 'transformers>=4.51.3' accelerate bitsandbytes 'huggingface_hub>=0.20' datasets pillow pandas numpy pyyaml

In [ ]:
# 2) Subir y descomprimir el proyecto (proyecto.zip con src/, config.yaml, ...)
from google.colab import files
import zipfile, os

uploaded = files.upload()  # selecciona proyecto.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/proyecto')
os.chdir('/content/proyecto')
print(os.listdir('.'))

In [ ]:
# 3) Login en Hugging Face (lee el secret HF_TOKEN de Colab)
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# 4) Descargar el dataset COMPLETO (incluye máscaras de disco, necesarias para el pooling roi)
#    y construir data/master_table.csv si no existe
!python -m src.data

In [ ]:
# 5) Extracción (~15–25 min). Prueba rápida primero con: --n 3
!python -m src.extract_pooling_maps --prompt P1

In [ ]:
# 6) Descargar el resultado
from google.colab import files
files.download('/content/proyecto/results/pooling_maps.csv')

# Después, en tu máquina local:
#   1. Copia pooling_maps.csv a results/pooling_maps.csv
#   2. Ejecuta: python app/prepare_assets.py
#   3. Reinicia el dashboard: aparecerá el tab «🗺️ Mapas de pooling»